In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# 1. Verification Gate
registry_file = "location_registry.csv"
nasa_file = "nasa_global_landslide_catalog.csv"

if not os.path.exists(registry_file) or not os.path.exists(nasa_file):
    raise FileNotFoundError("Prerequisite dimension sheets are missing from the folder path.")

df_registry = pd.read_csv(registry_file)
df_nasa = pd.read_csv(nasa_file)

In [3]:
# 2. High-Precision Spherical Haversine Distance Engine
def calculate_haversine_distance(lat1, lon1, lat2, lon2):
    """
    Computes the great-circle distance between two coordinate pairs on a 
    spherical Earth surface using pure numpy array mathematics.
    """
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2.0 * np.arcsin(np.sqrt(a))
    earth_radius_km = 6371.0
    return earth_radius_km * c

In [4]:
# 3. Spatial Processing Loop
nearest_distances = []
incident_densities = []

RADIUS_KM = 50.0
SEARCH_AREA_SQKM = np.pi * (RADIUS_KM ** 2)

In [5]:
print("--- EXECUTING EMPIRICAL GEOSPATIAL EXTRACTION ENGINE ---")

for index, reg_row in df_registry.iterrows():
    city = reg_row['location']
    c_lat, c_lon = reg_row['latitude'], reg_row['longitude']
    
    # Calculate spherical distances to every single event in the NASA Catalog
    all_incident_distances = []
    for _, nasa_row in df_nasa.iterrows():
        distance = calculate_haversine_distance(c_lat, c_lon, nasa_row['latitude'], nasa_row['longitude'])
        all_incident_distances.append(distance)
        
    # Metric 1: Distance to the nearest verified historical landslide event
    min_distance_km = min(all_incident_distances)
    
    # Metric 2: Density per 1000 sq km within our regional zone territory
    incidents_in_radius = sum(1 for d in all_incident_distances if d <= RADIUS_KM)
    density_metric = (incidents_in_radius / SEARCH_AREA_SQKM) * 1000.0
    
    nearest_distances.append(min_distance_km)
    incident_densities.append(density_metric)
    print(f"  -> {city}: Nearest catalog event is {min_distance_km:.2f} km away. Regional Density: {density_metric:.4f}")

--- EXECUTING EMPIRICAL GEOSPATIAL EXTRACTION ENGINE ---
  -> Manali: Nearest catalog event is 2.45 km away. Regional Density: 0.2546
  -> Shimla: Nearest catalog event is 2.01 km away. Regional Density: 0.1273
  -> Mussoorie: Nearest catalog event is 2.54 km away. Regional Density: 0.1273
  -> Nainital: Nearest catalog event is 1.64 km away. Regional Density: 0.1273
  -> Leh: Nearest catalog event is 1.14 km away. Regional Density: 0.1273
  -> Darjeeling: Nearest catalog event is 1.30 km away. Regional Density: 0.2546
  -> Goa: Nearest catalog event is 1.98 km away. Regional Density: 0.1273
  -> Jaipur: Nearest catalog event is 451.75 km away. Regional Density: 0.0000
  -> Munnar: Nearest catalog event is 0.44 km away. Regional Density: 0.1273
  -> Ooty: Nearest catalog event is 1.86 km away. Regional Density: 0.1273


In [6]:
# Append empirical spatial indicators directly to our dataframe
df_registry['nearest_landslide_km'] = nearest_distances
df_registry['landslide_density_per_1000sqkm'] = incident_densities

In [7]:
# 4. Save the Pristine Empirical Output Asset to Disk
df_registry.to_csv("landslide_features.csv", index=False)
print(" Notebook 02 Complete! Clean, non-synthetic matrix written as 'landslide_features.csv'.")

 Notebook 02 Complete! Clean, non-synthetic matrix written as 'landslide_features.csv'.


In [8]:
# Final verification preview of our clean features
df_registry[['location', 'destination_type', 'elevation', 'mountain_flag', 'coastal_flag', 'nearest_landslide_km', 'landslide_density_per_1000sqkm']]

,location,destination_type,elevation,mountain_flag,coastal_flag,nearest_landslide_km,landslide_density_per_1000sqkm
0,Manali,Mountain Hill Station,1864.0,1,0,2.450546,0.254648
1,Shimla,Mountain Hill Station,2195.0,1,0,2.013015,0.127324
2,Mussoorie,Mountain Hill Station,2006.0,1,0,2.539590,0.127324
3,Nainital,Mountain Hill Station,1938.0,1,0,1.644790,0.127324
4,Leh,High Altitude Desert,3414.0,1,0,1.138329,0.127324
5,Darjeeling,Mountain Hill Station,2003.0,1,0,1.295102,0.254648
6,Goa,Coastal Plain,98.0,0,1,1.984492,0.127324
7,Jaipur,Semi-Arid Plain,431.0,0,0,451.745272,0.000000
8,Munnar,Mountain Hill Station,1461.0,1,0,0.440143,0.127324
9,Ooty,Mountain Hill Station,2235.0,1,0,1.857387,0.127324
